# Inicialização

In [ ]:
%pip install selenium pandas

import os
import platform
import shutil
import socket
import subprocess
import time
import ctypes

import re
import json
from datetime import date

from pathlib import Path

from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys

port = 9222
profile = Path.home() / ".sefaz_automacao_perfil"
url = 'https://plataforma.sefaz.ma.gov.br/'

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
folder = fdoc() / 'Sefaz MA'
cache = folder / 'cache' / 'datas consultadas'
cache.parent.mkdir(parents = True, exist_ok = True)

interval = 60 * 60

def lcache():
    if cache.exists():
        try:
            return json.load(cache.read_text(encoding = 'utf-8'))
        except json.JSONDecodeError:
            return {}
    return {}

def scache(date):
    tmp = cache.with_suffix('.tmp')
    tmp.write_text(json.dumps(date, ensure_ascii = False,
                              indent = 2), encoding = 'utf-8')
    tmp.replace(cache)

def key(name, b, e):
    cnpj = re.search(r'CNPJ: [\d./-]+', name).group(1)
    return f'{cnpj}|{b}|{e}'

def need_consulting(k):
    last = lcache().get(k)
    return last is None or (time.time() - last) >= interval

def write_con(k):
    dates = lcache()
    dates[k] = time.time()
    scache(dates)

# Automatização

In [23]:
browser = obrowser()
wait = WebDriverWait(browser, timeout = 15)
print('Conectado via:', browser.current_url)




DevTools listening on ws://127.0.0.1:9222/devtools/browser/2922d702-1532-4394-9890-8b701b936a3d


Conectado via: https://plataforma.sefaz.ma.gov.br/#/


In [26]:
btt = (By.XPATH, '/html/body/app-root/app-main/div/app-topbar/div/a/i')

wait.until(EC.element_to_be_clickable(btt)).click()

btt = (By.XPATH, '/html/body/app-root/app-main/div/div[1]/app-menu/div/ul/li/ul/li[2]/a/span')

wait.until(EC.element_to_be_clickable(btt)).click()

ddfe = wait.until(EC.element_to_be_clickable((By.XPATH, '/html/body/app-root/app-main/div/div[1]/app-menu/div/ul/li/ul/li[2]/ul/li[1]/a/span')))

ddfe.click()

In [ ]:


firms = listlist()
print(f'{len(firms)} empresas encontradas')




156 empresas encontradas


"for i, name in enumerate(firms, 1):\n    try:\n        select(name)\n    except Exception as e:\n        print(f'Erro - Selecionar {name}: {e}')"

In [ ]:

now = date.today()
b = now.replace(day = 1)
e = now

b = b.isoformat()
e = e.isoformat()

for i, name in enumerate(firms, 1):
    k = key(name, b, e)

    if not need_consulting(k):
        print(f'[{i}/{len(firms)}] {name}: consultado há menos de 1h. Pulando...')

    m = str(now.month)
    y = str(now.year)

    try:
        select(name)

        square = (By.XPATH, '//*[@id="competencia"]')

        data = wait.until(EC.element_to_be_clickable(square)).click()
        data.send_keys(Keys.CONTROL, 'a')
        data.send_keys(Keys.DELETE)
        data.send_keys(m, y)

        tipos = ('NFE', 'NFCE')

        for tipo in tipos:
            print(f'Iniciando o download das {tipo}...')

            wait.until(
                EC.element_to_be_clickable(
                    (By.XPATH,
                     '/html/body/app-root/app-main/div/div[2]/div/app-download-form/p-card[1]/div/div/div[2]/form/div[3]/div[3]/div/div[2]/p-dropdown/div/span')
                     )).click()
            
            opcao = wait.until(EC.element_to_be_clickable((By.XPATH, f"//li[contains(@class,'p-dropdown-item')][@aria-label='{tipo}']")))

            wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="pr_id_4_label"]'))).click()

            wait.until(EC.element_to_be_clickable((By.XPATH, '///*[@id="p-highlighted-option"]'))).click()

            wait.until(EC.element_to_be_clickable((By.XPATH, '/html/body/app-root/app-main/div/div[2]/div/app-download-form/p-card[1]/div/div/div[2]/form/div[3]/div[7]/div/div[2]/div/button')))

    except Exception as e:
        print(f'Erro - Selecionar {name}: {e}')